In [1]:
import pandas as pd
import numpy as np
import os
import sys
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from openpyxl.styles import Font, Alignment, PatternFill
from openpyxl.utils import get_column_letter
import openai

# Precizna putanja do 'config' foldera
config_path = os.path.abspath('../../config')  # Relativna putanja -> apsolutna putanja

# Dodajemo 'config' folder u sys.path
if config_path not in sys.path:
    sys.path.append(config_path)

# Uvoz OPENAI_API_KEY iz config.py fajla
try:
    from config import OPENAI_API_KEY
    openai.api_key = OPENAI_API_KEY
    print("API ključ je uspešno učitan!")
except ImportError as e:
    print("Greška pri uvozu config.py: ", e)
except Exception as e:
    print("Došlo je do greške: ", e)

# Primer korišćenja OpenAI API sa novim modelom
def test_openai():
    try:
        response = openai.ChatCompletion.create(
            model="gpt-3.5-turbo",  # Preporučeni model
            messages=[
                {"role": "system", "content": "Ti si asistent za analizu podataka."},
                {"role": "user", "content": "Recite mi nešto o analizi podataka."}
            ]
        )
        print("Odgovor OpenAI API-ja:", response['choices'][0]['message']['content'].strip())
    except Exception as e:
        print("Došlo je do greške prilikom poziva OpenAI API-ja: ", e)

# Poziv funkcije za testiranje OpenAI API-ja
test_openai()


API ključ je uspešno učitan!
Odgovor OpenAI API-ja: Analiza podataka je proces obrade, provere, transformacije i modeliranja podataka radi otkrivanja korisnih informacija, prepoznavanja uzoraka, donošenja zaključaka i podrške donošenju odluka. Ova analiza može uključivati različite tehnike i alate, poput statističke analize, mašinskog učenja, analize teksta i vizualizacije podataka. Cilj analize podataka je pružiti uvid u poslovne procese, identifikovati trendove, predvideti buduće događaje i poboljšati poslovne performanse.


In [1]:
import pandas as pd
import numpy as np
import os
import sys
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from openpyxl.styles import Font, Alignment, PatternFill
from openpyxl.utils import get_column_letter
import openai
import ipywidgets

# Precizna putanja do 'config' foldera van 'notebooks' direktorijuma
config_path = '/home/malac/Desktop/MIT/projekat/validoAI/config/'

# Dodajemo 'config' folder u sys.path
sys.path.append(config_path)
# Putanja do foldera gde su XML fajlovi
folder_path = "../../data/mom21/pppd"

# Lista za čuvanje podataka iz XML fajlova
data = []

# Iteracija kroz sve XML fajlove u folderu
for file in os.listdir(folder_path):
    if file.endswith(".xml"):  # Provera da li je fajl XML
        file_path = os.path.join(folder_path, file)  # Kreiranje pune putanje do fajla
        tree = ET.parse(file_path)  # Parsiranje XML fajla
        root = tree.getroot()  # Dobijanje osnovnog elementa XML strukture

        # Pronalazak i čitanje podataka o prijavi
        for prijava in root.findall(".//{http://pid.purs.gov.rs}PodaciOPrijavi"):
            oznaka = prijava.find("{http://pid.purs.gov.rs}KlijentskaOznakaDeklaracije").text
            period = prijava.find("{http://pid.purs.gov.rs}ObracunskiPeriod").text
            datum = prijava.find("{http://pid.purs.gov.rs}DatumPlacanja").text
            
            # Pronalazak i čitanje ukupnih podataka
            for ukupno in root.findall(".//{http://pid.purs.gov.rs}Ukupno"):
                bruto = ukupno.find("{http://pid.purs.gov.rs}Bruto").text
                porez = ukupno.find("{http://pid.purs.gov.rs}Porez").text
                pio = ukupno.find("{http://pid.purs.gov.rs}PIO").text
                zdr = ukupno.find("{http://pid.purs.gov.rs}ZDR").text
                nez = ukupno.find("{http://pid.purs.gov.rs}NEZ").text

                # Dodavanje osnovnih podataka u listu
                data.append([oznaka, period, datum, bruto, porez, pio, zdr, nez])

            # Pronalazak i čitanje deklarisanih prihoda
            for prihod in root.findall(".//{http://pid.purs.gov.rs}DeklarisaniPrihodi"):
                for podaci in prihod.findall("{http://pid.purs.gov.rs}PodaciOPrihodima"):
                    ime = podaci.find("{http://pid.purs.gov.rs}Ime").text
                    prezime = podaci.find("{http://pid.purs.gov.rs}Prezime").text
                    bruto = podaci.find("{http://pid.purs.gov.rs}Bruto").text
                    porez = podaci.find("{http://pid.purs.gov.rs}Porez").text
                    pio = podaci.find("{http://pid.purs.gov.rs}PIO").text
                    zdr = podaci.find("{http://pid.purs.gov.rs}ZDR").text
                    nez = podaci.find("{http://pid.purs.gov.rs}NEZ").text
                    
                    # Dodavanje podataka o prihodima u listu
                    data.append([oznaka, period, datum, bruto, porez, pio, zdr, nez, ime, prezime])

# Kreiranje DataFrame-a iz prikupljenih podataka
columns = ["Oznaka deklaracije", "Obračunski period", "Datum plaćanja", 
           "Bruto", "Porez", "PIO", "ZDR", "NEZ", "Ime", "Prezime"]
df = pd.DataFrame(data, columns=columns)

# Stilizacija HTML tabele
styled_df = df.style.set_table_styles([
    {"selector": "thead th", "props": [("background-color", "#4f81bd"),
                                       ("color", "white"),
                                       ("text-align", "center"),
                                       ("font-size", "12pt"),
                                       ("border", "1px solid black")]},
    {"selector": "tbody td", "props": [("background-color", "#f9f9f9"),
                                       ("color", "black"),
                                       ("text-align", "center"),
                                       ("font-size", "10pt"),
                                       ("border", "1px solid black")]},
    {"selector": "tbody tr:nth-child(even) td", "props": [("background-color", "#eaf3f8")]}
]).set_properties(**{"border": "1px solid black", "width": "200px"})  # Dodata širina kolone


display(styled_df)  # Radi samo u Jupyter Notebook-u

# Čuvanje u Excel fajl sa stilizovanim zaglavljem
output_path = "../../data/mom21/pppd_analiza_finalno_stilizovano.xlsx"
with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    df.to_excel(writer, index=False, sheet_name="PPPD Podaci")
    workbook = writer.book
    worksheet = writer.sheets["PPPD Podaci"]

    # Stil za zaglavlje
    header_font = Font(bold=True, color="FFFFFF")  # Bold i bela boja fonta
    header_fill = PatternFill(start_color="4F81BD", end_color="4F81BD", fill_type="solid")  # Plava pozadina
    header_alignment = Alignment(horizontal="center", vertical="center")  # Centrirano poravnanje

    # Primena stila na zaglavlje
    for col_num, column_title in enumerate(df.columns, start=1):
        cell = worksheet.cell(row=1, column=col_num)
        cell.value = column_title
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = header_alignment

    # Podesavanje širine kolona za bolju čitljivost
    for col_num, column_cells in enumerate(worksheet.columns, start=1):
        max_length = max((len(str(cell.value)) for cell in column_cells if cell.value), default=0)
        worksheet.column_dimensions[get_column_letter(col_num)].width = max_length + 2

    #print(f"Podaci su uspešno sačuvani u {output_path}")


,Oznaka deklaracije,Obračunski period,Datum plaćanja,Bruto,Porez,PIO,ZDR,NEZ,Ime,Prezime
0,152497658,2024-09,2024-10-31,289995.69,18999.56,69598.96,29869.58,2174.97,None,None
1,152497658,2024-09,2024-10-31,83068.47,5806.85,19936.44,8556.06,623.01,MILOVAN,VASOVIC
2,152497658,2024-09,2024-10-31,82738.94,5773.89,19857.34,8522.12,620.54,DRAGAN,TRBOJEVIC
3,152497658,2024-09,2024-10-31,62094.14,3709.41,14902.59,6395.70,465.71,MARIJA,MARKOVIĆ
4,152497658,2024-09,2024-10-31,62094.14,3709.41,14902.59,6395.70,465.71,MARIJA,VASOVIC
5,147082135,2024-05,2024-07-05,218922.34,14392.24,52541.36,22548.98,1641.92,None,None
6,147082135,2024-05,2024-07-05,83006.45,5800.65,19921.55,8549.66,622.55,MILOVAN,VASOVIC
7,147082135,2024-05,2024-07-05,67698.32,4269.83,16247.59,6972.92,507.74,DRAGAN,TRBOJEVIC
8,147082135,2024-05,2024-07-05,68217.57,4321.76,16372.22,7026.40,511.63,MARIJA,MARKOVIĆ
9,151056190,2024-08,2024-09-30,281915.81,18191.59,67659.80,29037.32,2114.36,None,None
